# 🎙️ Kokoro-82M TTS — Text-to-Speech Audio Generator

> **Kaggle-Optimized Version** | GPU T4 x2 Accelerated
>
> Convert long-form articles (~8,000 words) into high-quality audiobook-style audio using the Kokoro-82M TTS engine.

---

### ⚙️ Setup Instructions
1. **Enable GPU**: Settings → Accelerator → **GPU T4 x2**
2. **Add input data**: Upload your `.txt` file via **Add Data** or place it in `/kaggle/input/`
3. **Run cells sequentially**: Step 1 → Step 2 → ... → Step 7

### 📂 Input File
- Place your article `.txt` file in `/kaggle/input/<dataset-name>/`
- Or set `INPUT_TEXT` variable directly in Cell 1

### 📁 Output
- Generated audio files are saved to `/kaggle/working/output/`
- Download via **Output** tab or the built-in download cell (Step 7)

In [ ]:
# ════════════════════════════════════════════════════════════
# ⚙️ CONFIGURATION — Edit these values as needed
# ════════════════════════════════════════════════════════════

# --- INPUT OPTIONS ---
# Paste text directly here (takes precedence over file)
INPUT_TEXT = ""

# Path to your .txt file in Kaggle input
# Example: "/kaggle/input/my-article/article.txt"
# Leave empty ("") to auto-detect .txt files in /kaggle/input/
INPUT_FILE_PATH = ""

# --- VOICE SELECTION (Official Kokoro-82M Voices) ---
# Language: 'a' = American English | 'b' = British English
LANGUAGE = "a"

# Voice options: af_heart (best), af_bella, af_nicole, af_sarah, af_kore,
#   af_alloy, af_aoede, af_nova, af_sky, af_river, af_jessica,
#   am_michael, am_puck, am_fenrir, am_echo, am_eric, am_liam,
#   am_onyx, am_adam, am_santa,
#   bf_emma, bf_isabella, bf_alice, bf_lily,
#   bm_fable, bm_george, bm_lewis, bm_daniel
VOICE = "af_heart"

# --- AUDIO QUALITY SETTINGS ---
SPEED = 0.92                    # 0.92 = Natural pace (~155 WPM)
PAUSE_BETWEEN_SECTIONS = 0.6    # Pause between chunks (seconds)
PAUSE_AT_HEADERS = 1.2          # Longer pause at section breaks
CHUNK_SIZE = 135                # Words per segment (120-150 optimal)

# --- OUTPUT SETTINGS ---
OUTPUT_FORMAT = "wav"           # wav (highest quality)
NORMALIZATION = "podcast"       # off, peak, acx, podcast (-19 LUFS)
ARTICLE_CODE = "E1"            # Your article code (e.g., A4, P2, E15)
ARTICLE_SLUG = "dartmouth-conference-1956"  # URL slug for filename

# --- ADVANCED OPTIONS ---
ENABLE_PRONUNCIATION_SYSTEM = False  # ⚠️ KEEP DISABLED (True breaks Kokoro)
GPU_ACCELERATION = True             # Use GPU for 3-4x speedup
SAVE_CHECKPOINTS = True             # Save progress incrementally
GENERATE_DOWNLOAD_DASHBOARD = False  # ⚠️ DISABLED (ngrok unavailable on Kaggle)

# ════════════════════════════════════════════════════════════
# DISPLAY CONFIGURATION SUMMARY
# ════════════════════════════════════════════════════════════

print("Configuration loaded!")
print()
print("CURRENT SETTINGS:")
print(f"   Voice:        {VOICE}")
print(f"   Speed:        {SPEED}x")
print(f"   Chunk size:   ~{CHUNK_SIZE} words per segment")
print(f"   Output:       {OUTPUT_FORMAT.upper()} format")
print(f"   Normalization: {NORMALIZATION}")
print(f"   Language:     {'American English' if LANGUAGE == 'a' else 'British English'}")
print()
print("ADVANCED SETTINGS:")
print(f"   Pronunciation: {'DISABLED (Safe!)' if not ENABLE_PRONUNCIATION_SYSTEM else 'ENABLED (Test first!)'}")
print(f"   GPU:           {'Enabled' if GPU_ACCELERATION else 'CPU only'}")
print(f"   Checkpoints:   {'Enabled' if SAVE_CHECKPOINTS else 'Disabled'}")
print(f"   Dashboard:     {'Skipped (Kaggle)' if not GENERATE_DOWNLOAD_DASHBOARD else 'Will generate'}")
print()

quality_info = {
    "af_heart": "GRADE A (Highest Quality!)",
    "af_bella": "GRADE A- (Excellent)",
    "af_nicole": "GRADE B- (Good)",
    "bf_emma": "GRADE B- (Good, British)",
    "am_michael": "GRADE C+ (Best Male)",
    "am_puck": "GRADE C+",
}

if VOICE in quality_info:
    print(f"VOICE QUALITY: {VOICE} -> {quality_info[VOICE]}")
else:
    print(f"VOICE QUALITY: {VOICE} -> See options above for grade")

print()
print("READY TO GO!")
print("   Next: Run Step 1 (Install), then Step 2 (Load Model), then Step 3 (Load Text)")


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 1: Install Dependencies
# ════════════════════════════════════════════════════════════

import sys
import subprocess

print("Installing Kokoro TTS and audio processing libraries...")
print("-" * 60)

# Core TTS engine
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
               "kokoro>=0.9.4", "soundfile", "pydub", "pyloudnorm"],
               check=True)

# Install CMU Pronouncing Dictionary (optional)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cmudict"],
               capture_output=True)

# System dependencies for audio processing (Kaggle has ffmpeg;
# espeak-ng may need install)
subprocess.run(["apt-get", "-qq", "-y", "install", "espeak-ng"],
               capture_output=True)

import torch

print(f"\nInstallation Complete!")
print(f"   Python: {sys.version.split()[0]}")
print(f"   PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU Detected: {gpu_name}")
    print(f"   VRAM Available: {gpu_vram:.1f} GB")
    print(f"   GPU Acceleration: {'Ready' if GPU_ACCELERATION else 'Disabled in settings'}")
else:
    print("   No GPU detected - will use CPU (slower but works)")
    print("   Tip: Settings -> Accelerator -> GPU T4 x2")

print("\n" + "=" * 60)
print("Additional packages installed:")
print("   - kokoro (TTS engine)")
print("   - soundfile (audio I/O)")
print("   - pydub (audio processing)")
print("   - pyloudnorm (loudness normalization)")
print("   - cmudict (pronunciation dictionary)")
print("   - espeak-ng (multi-language phonemes)")
print("=" * 60)


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 2: Load Kokoro-82M Model & Setup
# ════════════════════════════════════════════════════════════

# GLOBAL IMPORTS
import os
import re
import gc
import time
import subprocess
import numpy as np
import torch
import soundfile as sf
from datetime import datetime, timedelta
from IPython.display import clear_output, display, Markdown, Audio

# CONFIGURE DEVICE & LOAD KOKORO PIPELINE
print("Setting up Kokoro-82M TTS engine...")
print("-" * 60)

if GPU_ACCELERATION and torch.cuda.is_available():
    device = "cuda"
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = "cpu"
    print("Using CPU (slower but works)")

from kokoro import KPipeline

# Kokoro native sample rate
SR = 24000

print("\nLoading Kokoro-82M model...")
print("   This may take 15-30 seconds on first run (downloading model weights)")

pipeline = KPipeline(lang_code=LANGUAGE)

# Warm-up generation to load model into GPU memory
try:
    generator = pipeline("Test.", voice=VOICE, speed=SPEED)
    for _ in generator:
        pass
    print("Model loaded and ready!")
except Exception as e:
    print(f"Warm-up issue (non-critical): {e}")

# ════════════════════════════════════════════════════════════
# OPTIONAL PRONUNCIATION SYSTEM
# ════════════════════════════════════════════════════════════
class PronunciationResolver:
    """
    Optional pronunciation enhancement using CMUdict + eSpeak fallback.
    Note: Kokoro handles most English words perfectly; use only if necessary.
    """
    def __init__(self):
        self.stats = {}
        try:
            import cmudict
            self.cmu = cmudict.dict()
            self.stats['cmudict_loaded'] = len(self.cmu)
        except:
            self.cmu = {}
            print("   CMUdict not available")

    def resolve(self, text):
        # Placeholder: returns text unchanged to avoid breaking Kokoro
        return text

    def get_stats(self):
        return self.stats

if ENABLE_PRONUNCIATION_SYSTEM:
    print("\nInitializing pronunciation system...")
    pronouncer = PronunciationResolver()
    print("   (Pronunciation enhancement is ENABLED)")
else:
    pronouncer = None
    print("\nPronunciation system: DISABLED (using Kokoro's native handling)")

# ════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ════════════════════════════════════════════════════════════
def format_time(seconds):
    """Convert seconds to HH:MM:SS format"""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h}:{m:02d}:{s:02d}"

print("\n" + "=" * 60)
print("AVAILABLE VOICES (Kokoro-82M)")
print("=" * 60)
print(f"Selected: {VOICE}")
print(f"Language: {'American English' if LANGUAGE == 'a' else 'British English'}")
print(f"Sample rate: {SR} Hz")
print(f"Device: {device.upper()}")
print("\nReady! Proceed to Step 3 to load your article text.")


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 3: Load Your Article
# ════════════════════════════════════════════════════════════

import glob

text_content = None
filename = None

# Priority 1: Direct text input
if INPUT_TEXT.strip():
    text_content = INPUT_TEXT.strip()
    filename = "direct_input"
    print("Using text from INPUT_TEXT variable.")

# Priority 2: Explicit file path
elif INPUT_FILE_PATH.strip():
    if os.path.exists(INPUT_FILE_PATH):
        with open(INPUT_FILE_PATH, 'r', encoding='utf-8') as f:
            text_content = f.read().strip()
        filename = os.path.basename(INPUT_FILE_PATH)
        print(f"Loaded file from: {INPUT_FILE_PATH}")
    else:
        raise FileNotFoundError(
            f"File not found: {INPUT_FILE_PATH}\n\n"
            f"Make sure your .txt file is added as a dataset."
        )

# Priority 3: Auto-detect .txt files in /kaggle/input/
else:
    txt_files = glob.glob("/kaggle/input/**/*.txt", recursive=True)
    if not txt_files:
        raise FileNotFoundError(
            "No .txt file found!\n\n"
            "Please do one of the following:\n"
            "  1. Set INPUT_TEXT in the Configuration cell\n"
            "  2. Set INPUT_FILE_PATH to the full path of your .txt file\n"
            "  3. Add a dataset containing a .txt file via Add Data"
        )

    # Use the first .txt file found
    selected_file = txt_files[0]
    with open(selected_file, 'r', encoding='utf-8') as f:
        text_content = f.read().strip()
    filename = os.path.basename(selected_file)

    print(f"Auto-detected text file:")
    print(f"   {selected_file}")
    if len(txt_files) > 1:
        print(f"   (Found {len(txt_files)} .txt files, using the first one)")
        print("   To use a specific file, set INPUT_FILE_PATH in Configuration.")

# Basic validation
word_count = len(text_content.split())
char_count = len(text_content)

if word_count < 100:
    raise ValueError(
        f"File too short: {word_count} words\n\n"
        f"Please upload the complete article (expected ~8,000 words)."
    )

# Estimate duration
est_wpm = 155 if SPEED <= 0.95 else 170
est_minutes = word_count / est_wpm
est_chunks = max(1, word_count // CHUNK_SIZE)

print(f"\nFile loaded successfully!\n")
print(f"Filename: {filename}")
print(f"Word count: {word_count:,}")
print(f"Character count: {char_count:,}")
print(f"Est. audio duration: ~{est_minutes:.1f} minutes")
print(f"Est. segments: ~{est_chunks}")
print(f"Voice: {VOICE} | Speed: {SPEED}x")
print(f"Format: {OUTPUT_FORMAT.upper()} | Normalize: {NORMALIZATION}")

# Apply pronunciation enhancement if enabled (with safety check)
if ENABLE_PRONUNCIATION_SYSTEM and pronouncer is not None:
    print("\nApplying pronunciation enhancement...")
    text_content = pronouncer.resolve(text_content)
    stats = pronouncer.get_stats()
    if stats:
        print("   Resolution stats:")
        for source, count in stats.items():
            print(f"      - {source}: {count}")

print("\n" + "=" * 60)
print("Ready for generation! Proceed to Step 4.")
print("=" * 60)


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 4: Generate Audio
# ════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════
# INTELLIGENT TEXT CHUNKER
# ════════════════════════════════════════════════════════════

def chunk_text_intelligently(text, max_words=CHUNK_SIZE):
    """
    Split text into chunks at natural speech boundaries.

    Strategy:
    1. Split by double newlines (paragraphs) first
    2. If paragraph too long, split by sentences (. ! ?)
    3. Find nearest sentence boundary to target word count
    4. Never split mid-sentence (critical for TTS quality!)
    5. Validate chunks are within acceptable size range
    """

    # Pre-process: normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    chunks = []
    current_chunk = []
    current_count = 0

    for para in paragraphs:
        para_words = para.split()

        # If paragraph fits entirely, keep it whole
        if len(para_words) <= max_words:
            if current_count + len(para_words) > max_words and current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = para_words
                current_count = len(para_words)
            else:
                current_chunk.extend(para_words)
                current_count += len(para_words)
        else:
            # Paragraph too long - split by sentences
            if current_chunk:
                chunks.append(' '.join(current_chunk))
                current_chunk = []
                current_count = 0

            # Sentence splitting
            sentences = re.split(r'(?<=[.!?])\s+', para)

            for sent in sentences:
                sent_words = sent.split()

                if current_count + len(sent_words) > max_words:
                    if current_chunk:
                        chunks.append(' '.join(current_chunk))
                        current_chunk = []
                        current_count = 0

                    # Single sentence too long? Force split (rare)
                    if len(sent_words) > max_words:
                        mid = len(sent_words) // 2
                        best_break = max_words
                        for i in range(max(0, mid-50), min(len(sent_words), mid+50)):
                            if sent_words[i].endswith(('.', '!', '?')):
                                best_break = i + 1
                                break

                        part1 = sent_words[:best_break]
                        part2 = sent_words[best_break:]

                        chunks.append(' '.join(part1))
                        current_chunk = part2
                        current_count = len(part2)
                    else:
                        current_chunk = sent_words
                        current_count = len(sent_words)
                else:
                    current_chunk.extend(sent_words)
                    current_count += len(sent_words)

    # Don't forget last chunk!
    if current_chunk:
        chunks.append(' '.join(current_chunk))

    # Filter out tiny chunks (< 30 words) by merging with neighbors
    final_chunks = []
    i = 0
    while i < len(chunks):
        chunk = chunks[i]
        if len(chunk.split()) < 30 and i < len(chunks) - 1:
            chunk = chunk + ' ' + chunks[i + 1]
            i += 1
        final_chunks.append(chunk)
        i += 1

    return final_chunks


def detect_section_header(text):
    """Detect if chunk starts with a header (## or ### style)"""
    lines = text.strip().split('\n')
    for line in lines[:3]:
        line = line.strip()
        if line.startswith('##') or re.match(r'^#{1,3}\s', line):
            header = re.sub(r'^#+\s*', '', line)
            header = re.sub(r'[#*_`]', '', header).strip()
            return header[:80] if header else None
    return None


# ════════════════════════════════════════════════════════════
# AUDIO PROCESSING UTILITIES
# ════════════════════════════════════════════════════════════

def normalize_audio(audio, mode=NORMALIZATION, sr=SR):
    """
    Apply loudness normalization.
    Handles large audio files safely with proper array shaping.
    Falls back gracefully if pyloudnorm fails.
    """
    if mode == "off" or mode is None:
        return audio

    if mode == "peak":
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio * (0.95 / peak)
            print(f"   Peak normalized (max amplitude: 0.95)")
        return audio

    try:
        import pyloudnorm as pyln

        print(f"   Calculating loudness...")
        meter = pyln.Meter(sr, block_size=0.400)

        if audio.ndim == 1:
            audio_2d = audio[:, np.newaxis]
        else:
            audio_2d = audio

        loudness = meter.integrated_loudness(audio_2d)

        print(f"   Measured loudness: {loudness:.1f} LUFS")

        targets = {
            "acx": -16.0,
            "podcast": -19.0,
            "youtube": -14.0,
        }

        try:
            target = float(mode)
        except ValueError:
            target = targets.get(mode.lower(), -19.0)

        print(f"   Target loudness: {target} LUFS ({mode})")

        if loudness > -70 and not np.isnan(loudness):
            audio_normalized = pyln.normalize.loudness(
                audio_2d, loudness, target
            )
            audio = audio_normalized.flatten()
            print(f"   Loudness normalization applied successfully!")

            new_loudness = meter.integrated_loudness(audio[:, np.newaxis])
            print(f"   New loudness: {new_loudness:.1f} LUFS")
        else:
            print(f"   Audio too quiet or invalid loudness ({loudness:.1f} LUFS)")
            print("      Falling back to peak normalization...")
            peak = np.max(np.abs(audio))
            if peak > 0:
                audio = audio * (0.95 / peak)

        return audio

    except Exception as e:
        print(f"   Loudness normalization failed: {e}")
        print("      Using peak normalization instead...")

        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio * (0.95 / peak)
            print(f"   Peak normalized successfully (fallback)")

        return audio


# ════════════════════════════════════════════════════════════
# MAIN GENERATION LOOP
# ════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("AUDIO STUDIO - GENERATION ENGINE")
print("=" * 70)
print(f"\nDevice: {device.upper()}")
print(f"Words: {len(text_content.split()):,}")
print(f"Voice: {VOICE} | Speed: {SPEED}x | Chunk size: {CHUNK_SIZE}")
print(f"Normalization: {NORMALIZATION} | Format: {OUTPUT_FORMAT.upper()}")
print()

# Prepare chunks
chunks = chunk_text_intelligently(text_content, max_words=CHUNK_SIZE)
total_chunks = len(chunks)

print(f"Created {total_chunks} segments for processing\n")
print("Starting generation...\n")

# Generation state
all_audio_segments = []
timestamps = []
current_time = 0.0
failed_chunks = []
start_time = time.time()

MAX_RETRIES = 3

# Create output directory (Kaggle working directory)
output_dir = "/kaggle/working/output"
os.makedirs(output_dir, exist_ok=True)

# Save checkpoints directory
checkpoint_dir = os.path.join(output_dir, "checkpoints")
if SAVE_CHECKPOINTS:
    os.makedirs(checkpoint_dir, exist_ok=True)

# Main processing loop
for idx, chunk_text in enumerate(chunks):
    chunk_start_time = current_time
    chunk_words = len(chunk_text.split())
    is_header = detect_section_header(chunk_text) is not None

    pause_duration = PAUSE_AT_HEADERS if is_header else PAUSE_BETWEEN_SECTIONS

    # Progress display
    pct_complete = (idx + 1) / total_chunks * 100
    elapsed = time.time() - start_time

    clear_output(wait=True)
    print(f"Generating Audio: {pct_complete:.1f}% Complete")
    print(f"  Progress: Segment {idx+1} of {total_chunks}")
    print(f"  Words this segment: {chunk_words:,}")
    print(f"  Elapsed: {format_time(elapsed)}")
    print(f"  Audio generated: {format_time(current_time)} so far")
    print(f"  Status: {'Processing...' if idx < total_chunks - 1 else 'Finalizing...'}")
    print(f"  Failures: {len(failed_chunks)} segments")
    print(f"  Preview: \"{chunk_text[:80]}...\"")

    # Generate audio with retry logic
    audio_np = None
    success = False

    for attempt in range(MAX_RETRIES):
        try:
            generator = pipeline(chunk_text, voice=VOICE, speed=SPEED)
            for result in generator:
                audio_np = result[2]

                if hasattr(audio_np, 'numpy'):
                    audio_np = audio_np.numpy().astype(np.float32)
                else:
                    audio_np = np.array(audio_np, dtype=np.float32).flatten()

                if len(audio_np) > 0:
                    success = True
                    break

            if success:
                break

        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait_time = (attempt + 1) * 2
                print(f"  Attempt {attempt+1} failed: {e}")
                print(f"  Retrying in {wait_time}s...")
                time.sleep(wait_time)

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    gc.collect()
            else:
                print(f"  Chunk {idx+1} failed after {MAX_RETRIES} attempts: {e}")

    # Handle failure
    if not success or audio_np is None or len(audio_np) == 0:
        failed_chunks.append(idx + 1)
        silence_samples = int((chunk_words / 155) * SR)
        audio_np = np.zeros(silence_samples, dtype=np.float32)
        print(f"  Segment {idx+1}: FAILED -> replaced with silence")

    # Calculate duration
    segment_duration = len(audio_np) / SR

    # Store audio
    all_audio_segments.append(audio_np)

    # Add pause after segment (except last)
    if idx < total_chunks - 1 and pause_duration > 0:
        pause_samples = int(pause_duration * SR)
        pause_audio = np.zeros(pause_samples, dtype=np.float32)
        all_audio_segments.append(pause_audio)
        segment_duration += pause_duration

    current_time += segment_duration

    # Detect title for timestamp
    title = detect_section_header(chunk_text)
    if not title:
        first_sent = re.split(r'(?<=[.!?])\s+', chunk_text, maxsplit=1)[0]
        title = first_sent[:60] + ("..." if len(first_sent) > 60 else "")

    timestamps.append((chunk_start_time, current_time, title))

    # Save checkpoint every 10 segments
    if SAVE_CHECKPOINTS and (idx + 1) % 10 == 0:
        checkpoint_file = os.path.join(checkpoint_dir, f"segment_{idx+1:04d}.wav")
        sf.write(checkpoint_file, audio_np, SR)

    # GPU cleanup periodically
    if (idx + 1) % 10 == 0 and torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()


# ════════════════════════════════════════════════════════════
# ASSEMBLE FINAL AUDIO
# ════════════════════════════════════════════════════════════

clear_output(wait=True)
print("Assembling Final Audio...\n")

print("Merging segments...")
merged = np.concatenate(all_audio_segments)
del all_audio_segments
gc.collect()

raw_duration = len(merged) / SR
print(f"Raw audio: {format_time(raw_duration)} ({raw_duration/60:.1f} min)")

# Normalize
print(f"Normalizing ({NORMALIZATION})...")
merged = normalize_audio(merged, mode=NORMALIZATION, sr=SR)

# Generate filename
timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
safe_slug = ARTICLE_SLUG.replace('/', '-').replace('\\', '-')
output_filename = f"{ARTICLE_CODE}-{safe_slug}-{timestamp_str}.{OUTPUT_FORMAT}"
output_filepath = os.path.join(output_dir, output_filename)

# Export
print(f"Exporting as {OUTPUT_FORMAT.upper()}...")
sf.write(output_filepath, merged, SR, subtype='PCM_24')

file_size_mb = os.path.getsize(output_filepath) / (1024 * 1024)
final_duration = current_time

# Calculate statistics
word_count = len(text_content.split())
total_processing_time = time.time() - start_time
words_per_minute = word_count / (final_duration / 60) if final_duration > 0 else 0

# ════════════════════════════════════════════════════════════
# GENERATE CHAPTER MARKERS FILE
# ════════════════════════════════════════════════════════════

chapters_file = os.path.join(output_dir, "chapters.txt")
with open(chapters_file, 'w', encoding='utf-8') as f:
    f.write(f"# Chapter Markers - {output_filename}\n\n")
    for start, end, title in timestamps:
        f.write(f"{format_time(start)} - {end:.1f}s | {title}\n")

# ════════════════════════════════════════════════════════════
# CLEANUP
# ════════════════════════════════════════════════════════════

del merged
gc.collect()

elapsed_final = time.time() - start_time

# ════════════════════════════════════════════════════════════
# DISPLAY FINAL REPORT
# ════════════════════════════════════════════════════════════

clear_output(wait=True)

status = "SUCCESS" if len(failed_chunks) == 0 else "WARNING (some segments failed)"

print("=" * 70)
print(f"Generation Complete! - {status}")
print("=" * 70)
print()
print("OUTPUT FILE")
print(f"  Filename:    {output_filename}")
print(f"  Location:    {output_filepath}")
print(f"  Size:        {file_size_mb:.1f} MB")
print(f"  Format:      {OUTPUT_FORMAT.upper()} (24-bit WAV)")
print(f"  Duration:    {format_time(final_duration)}")
print()
print("PERFORMANCE")
print(f"  Processing time:  {elapsed_final/60:.1f} min")
print(f"  Realtime ratio:   {final_duration/elapsed_final:.1f}x")
print(f"  Words per minute: {words_per_minute:.0f} WPM")
print()
print("QUALITY")
print(f"  Total segments:   {total_chunks}")
print(f"  Failed segments:  {len(failed_chunks)}")
print(f"  Normalization:    {NORMALIZATION}")
print()

if failed_chunks:
    print(f"  Failed segment numbers: {failed_chunks}")
    print("  (These were replaced with silence - check the audio)")
    print()

print("=" * 70)
print(f"DONE in {elapsed_final/60:.1f} minutes")
print(f"Audio: {output_filename} ({file_size_mb:.1f} MB)")
print(f"Duration: {format_time(final_duration)}")
print("=" * 70)
print()
print("Output files are in /kaggle/working/output/")
print("Download them from the Output tab on the right panel.")


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 5: Preview Audio
# ════════════════════════════════════════════════════════════

print("Creating audio preview...")
print("-" * 60)

try:
    # Read the generated file
    audio_data, sr = sf.read(output_filepath)

    # If stereo, take first channel
    if audio_data.ndim > 1:
        audio_data = audio_data[:, 0]

    print(f"Audio: {output_filename}")
    print(f"Duration: {len(audio_data)/sr:.1f} seconds")
    print(f"Sample rate: {sr} Hz")
    print()
    print("Preview (first 30 seconds):")

    preview_samples = min(len(audio_data), 30 * sr)
    preview_audio = audio_data[:preview_samples]

    display(Audio(preview_audio, rate=sr))

except Exception as e:
    print(f"Error creating preview: {e}")
    print()
    print("Alternative: Download the file from /kaggle/working/output/")


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 6: View Chapters & Metadata
# ════════════════════════════════════════════════════════════

print("Chapter Markers & Metadata")
print("-" * 60)
print()

# Display chapter markers
if os.path.exists(chapters_file):
    with open(chapters_file, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("No chapters file found.")

print()
print("-" * 60)
print("Output directory contents:")
total_size = 0
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024*1024)
        total_size += size
        print(f"   {f} ({size:.2f} MB)")

print(f"\n   Total size: {total_size / 1024:.2f} GB")


In [ ]:
# ════════════════════════════════════════════════════════════
# Step 7: Save Output for Download
# ════════════════════════════════════════════════════════════

# On Kaggle, output files in /kaggle/working/ are automatically
# available for download from the Output tab.
# This cell also creates a convenient compressed archive.

import shutil

print("=" * 70)
print("PREPARING OUTPUT FOR DOWNLOAD")
print("=" * 70)
print()

# List all output files
output_files = []
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024 * 1024)
        output_files.append((f, size))
        print(f"  {f} ({size:.1f} MB)")

print(f"\n  Total: {sum(s for _, s in output_files):.1f} MB")
print()

# Create a ZIP archive of all output for easy download
archive_name = f"kokoro-tts-output-{timestamp_str}.zip"
archive_path = f"/kaggle/working/{archive_name}"

shutil.make_archive(
    base_name=archive_path.replace('.zip', ''),
    format='zip',
    root_dir=output_dir
)

archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f"Archive created: {archive_name} ({archive_size:.1f} MB)")
print()
print("=" * 70)
print("HOW TO DOWNLOAD")
print("=" * 70)
print()
print("Option 1 (Recommended):")
print("  1. Click the 'Output' tab on the right panel")
print("  2. Find the .zip file and click Download")
print()
print("Option 2:")
print("  1. After the notebook finishes, click 'Save & Run All'")
print("  2. Go to your Kaggle profile -> Code -> this notebook")
print("  3. Click 'Output' to see and download all generated files")
print()
print("Option 3 (individual files):")
print("  1. Open the /kaggle/working/output/ folder in the file browser")
print("  2. Right-click any file to download")
print()
print("=" * 70)
print("DONE")
print("=" * 70)
